In [28]:
import os
from dotenv import load_dotenv
from typing import Annotated
from typing_extensions import TypedDict

from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain.messages import SystemMessage
from langgraph.graph import StateGraph, START,END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command
from langchain.messages import AIMessage,HumanMessage,SystemMessage
import requests

load_dotenv()

True

In [29]:
import smtplib
from email.message import EmailMessage


sender_email = "azrapatvi@gmail.com"
app_password = os.getenv('GMAIL_APP_PASSWORD')
receiver_email = "azrapatvi1234@gmail.com"

# Create email
msg = EmailMessage()
msg["Subject"] = "Test Email"
msg["From"] = sender_email
msg["To"] = receiver_email

msg.set_content("Hello! This email was sent using Python.")

# Send email
with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
    server.login(sender_email, app_password)
    server.send_message(msg)

print("Email sent successfully!")

Email sent successfully!


In [30]:
llm = ChatGroq(model="openai/gpt-oss-120b")

In [62]:
sender_email = "azrapatvi@gmail.com"
app_password = os.getenv('GMAIL_APP_PASSWORD')


@tool
def send_email(recipient:str,subject:str,body:str):
    """responsible for sending the email to the provided recipient along with subject and body"""
    decision=interrupt({
        "question":f"should i send the email to {recipient} with this {subject} and {body}?",
        "options":['yes','no']
    })

    if decision.lower().strip() == "yes":
        

        # Create email
        msg = EmailMessage()
        msg["Subject"] = subject
        msg["From"] = sender_email
        msg["To"] = recipient

        msg.set_content(body)

        # Send email
        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
            server.login(sender_email, app_password)
            server.send_message(msg)

        print("Email sent successfully!")
        return {
            "status":"success",
            "message":f"sent the email to {recipient} with this {subject} and {body}",
            "recipient":recipient,
            "subject":subject,
            "body":body
        }
    else:
        return {
                "status":"cancelled",
                "message":f"Failed to send theemail to {recipient} with this {subject} and {body}",
                "recipient":recipient,
                "subject":subject,
                "body":body
            }


In [63]:
tools=[send_email]

llm_with_tools=llm.bind_tools(tools)

In [64]:
class State(TypedDict):
    
    messages:Annotated[list,add_messages]

In [65]:
builder=StateGraph(State)

def chatbot(state:State):
    return {
        "messages":[llm_with_tools.invoke(state['messages'])]
    }

builder.add_node('chatbot',chatbot)
builder.add_node('tools',ToolNode(tools))

builder.add_edge(START,'chatbot')
builder.add_conditional_edges('chatbot',tools_condition)
builder.add_edge('tools','chatbot')

memory=InMemorySaver()
#compile the grpah
graph=builder.compile(checkpointer=memory)

In [66]:
config1={'configurable':{'thread_id':"chat1"}}
res=graph.invoke({"messages":"send email to azrapatvi1234@gmail.com with subject timepass and body hello"},config=config1)

In [67]:
res

{'messages': [HumanMessage(content='send email to azrapatvi1234@gmail.com with subject timepass and body hello', additional_kwargs={}, response_metadata={}, id='278c877d-67b4-4257-8970-8e890e6d172a'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email using function. Use provided function.', 'tool_calls': [{'id': 'fc_f54f2e2e-7477-449b-86e6-11bd5cebb7a2', 'function': {'arguments': '{"body":"hello","recipient":"azrapatvi1234@gmail.com","subject":"timepass"}', 'name': 'send_email'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 154, 'total_tokens': 216, 'completion_time': 0.130134313, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.006215838, 'prompt_tokens_details': None, 'queue_time': 0.343026629, 'total_time': 0.136350151}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_fe269835c7', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': Non

In [68]:
final_res = graph.invoke(
    Command(resume="yes"),
    config=config1
)

print(final_res)

Email sent successfully!
{'messages': [HumanMessage(content='send email to azrapatvi1234@gmail.com with subject timepass and body hello', additional_kwargs={}, response_metadata={}, id='278c877d-67b4-4257-8970-8e890e6d172a'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email using function. Use provided function.', 'tool_calls': [{'id': 'fc_f54f2e2e-7477-449b-86e6-11bd5cebb7a2', 'function': {'arguments': '{"body":"hello","recipient":"azrapatvi1234@gmail.com","subject":"timepass"}', 'name': 'send_email'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 154, 'total_tokens': 216, 'completion_time': 0.130134313, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.006215838, 'prompt_tokens_details': None, 'queue_time': 0.343026629, 'total_time': 0.136350151}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_fe269835c7', 'service_tier': 'on_demand', 'finish_reason': 'tool_

In [70]:
from langgraph.types import Command

config2 = {
    "configurable": {
        "thread_id": "chat_no_test"
    }
}

# Step 1: Start the email request
result = graph.invoke(
    {
        "messages": [
            ("user", "Send an email to azrapatvi1234@gmail.com with subject timepass and body hello")
        ]
    },
    config=config2
)

# Step 2: Reject the email
result = graph.invoke(
    Command(resume="no"),
    config=config2
)

# Step 3: Print all messages to inspect the tool result
for message in result["messages"]:
    print("\nType:", type(message).__name__)
    print("Content:", message.content)


Type: HumanMessage
Content: Send an email to azrapatvi1234@gmail.com with subject timepass and body hello

Type: AIMessage
Content: 

Type: ToolMessage
Content: {"status": "cancelled", "message": "Failed to send theemail to azrapatvi1234@gmail.com with this timepass and hello", "recipient": "azrapatvi1234@gmail.com", "subject": "timepass", "body": "hello"}

Type: AIMessage
Content: I attempted to send the email, but the operation was cancelled and the message could not be delivered. If you’d like to try again or need assistance with something else, just let me know!
